In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("silver.nyc_taxi_clean")

daily_revenue = silver_df.groupBy(
    "pickup_date"
).agg(
    round(sum("total_amount"), 2).alias("daily_revenue")
)

display(daily_revenue)

### Creating gold schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
(daily_revenue.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.daily_revenue"))

In [0]:
daily_trip_count = silver_df.groupBy(
    "pickup_date"
).agg(
    count("*").alias("total_trips")
)
display(daily_trip_count)

In [0]:
top_pickup_zones = silver_df.groupBy(
    "PULocationID"
).agg(
    count("*").alias("trip_count")
).orderBy(desc("trip_count"))

display(top_pickup_zones)

In [0]:
payment_analysis = silver_df.groupBy(
    "payment_type"
).agg(
    count("*").alias("trip_count"),
    round(sum("total_amount"), 2).alias("total_revenue")
)

display(payment_analysis)

In [0]:
%sql
DESCRIBE HISTORY silver.nyc_taxi_clean;